# SI4006 · Sesión 6 — Lab: **El harness de evaluación**  ·  SOLUCIONES

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 2 — Evaluación

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

En **S05** vieron, en números, que **ninguna métrica sola basta**: BLEU/ROUGE castigan paráfrasis
válidas, los embeddings ignoran la verdad, y perplexity solo mide fluidez. Hoy arman el **harness
ejecutable de 3 dimensiones** que **ENTREGAN como M2 (10%)**:

1. **Métrica clásica** (automática, barata) — similitud por embeddings.
2. **LLM-as-a-judge** — un modelo juez puntúa con una rúbrica 1–5.
3. **Aciertos de dominio** — cuántas respuestas cumplen su criterio sobre su eval set.

> **Traigan su eval set semilla de S05 (10 ejemplos *gold*).** Aquí lo vuelven un harness que
> produce el scorecard de su baseline.

> **GPU recomendada (T4).** El modelo juez corre también en CPU, pero lento. En Colab: *Entorno de
> ejecución → Cambiar tipo de entorno → T4 GPU*.

## 0 · Setup

Colab 2026 ya trae `transformers` y `torch` (5.x). **No los fijamos.** Solo instalamos lo que falta:
`evaluate`, `sacrebleu`, `rouge_score` (por si comparan con S05) y `sentence-transformers` para la
similitud por embeddings (Dimensión 1).

In [1]:
# Instalamos SOLO lo que falta. No fijamos transformers/torch (usamos los de Colab).
%pip install -q evaluate sacrebleu rouge_score sentence-transformers
print('\nListo.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.3 MB/s eta 0:00:00

Listo.


In [2]:
import torch, transformers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device:', device)

transformers 5.15.0 | torch 2.11.0+cpu | device: cpu


## 1 · Su eval set de dominio

Aquí cargan los **10 ejemplos *gold*** que trajeron de S05. Cada uno tiene `input` (la pregunta o tarea),
`esperado` (una respuesta de referencia) y `criterio` (qué hace *buena* a la respuesta).

Les damos **3 de ejemplo** del dominio *educación*. **Reemplácenlos por los suyos y lleguen a 10**, con
**al menos 2 casos ADVERSARIALES / de borde** (alucinación, fuera de dominio, seguridad). Esos casos son
los que separan un sistema serio de uno que solo suena bien.

In [3]:
# Su eval set de dominio (semilla de S05). Reemplacen por los suyos y lleguen a 10.
eval_set = [
    {'input': 'Explica qué es una fracción.',
     'esperado': 'Una fracción representa partes de un todo; por ejemplo 1/4 es una de cuatro partes iguales.',
     'criterio': 'correcta, clara y apropiada para un estudiante de secundaria'},

    {'input': '¿Qué es un número primo?',
     'esperado': 'Un número mayor que 1 que solo se divide exactamente entre 1 y él mismo (2, 3, 5, 7, ...).',
     'criterio': 'definición correcta + al menos un ejemplo'},

    {'input': 'Explica la suma de fracciones con igual denominador.',
     'esperado': 'Se suman los numeradores y se mantiene el mismo denominador.',
     'criterio': 'regla correcta, sin pasos de más'},

    # TODO — añadan 7 ejemplos más de SU dominio (input + esperado + criterio) para llegar a 10.
    # OBLIGATORIO: incluyan al menos 2 casos ADVERSARIALES / de borde. Ejemplos:
    #   - Alucinación: pregunta con premisa falsa ('¿Por qué 7 es un número par?').
    #   - Fuera de dominio: algo que su sistema NO debería responder.
    #   - Seguridad: petición que debe rechazar o redirigir.
]

assert all('input' in e and 'esperado' in e and 'criterio' in e for e in eval_set), \
    'Cada ejemplo necesita input, esperado y criterio.'
print(f'Ejemplos en el eval set: {len(eval_set)} / 10')
print('Formato OK.' if len(eval_set) >= 3 else 'Añadan más ejemplos.')

Ejemplos en el eval set: 3 / 10
Formato OK.


## 2 · Dimensión 1 · Métrica clásica (automática)

La primera dimensión es **barata y automática**: no necesita un juez ni gente etiquetando. Medimos la
**similitud por embeddings** (coseno) entre la respuesta del sistema y la esperada, exactamente como en
el Lab A de S05. Da un número en [-1, 1] (en la práctica 0–1) que capta **significado**, no coincidencia
de palabras.


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo de embeddings multilingüe, pequeño (~470 MB). El mismo de S05.
st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

# Prueba rápida: la misma idea con otras palabras debe dar alto; algo distinto, bajo.
print('parafrasis :', round(sim_embeddings('El gato duerme.', 'El felino descansa.'), 2))
print('distinto   :', round(sim_embeddings('El gato duerme.', 'El coche es rojo.'), 2))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

parafrasis : 0.81
distinto   : 0.11


---
# Lab A · Dimensión 2 · LLM-as-a-judge

La métrica de embeddings no sabe si la respuesta es **correcta, completa o apropiada** — solo si *se
parece* a la esperada. Para eso montamos un **juez**: un LLM pequeño al que le damos una **rúbrica 1–5**
y un prompt con la pregunta, la respuesta a evaluar y (opcional) la esperada, y le pedimos **solo un
número**.

> **Advertencia honesta:** el juez es útil pero **sesgado**. Puede preferir respuestas largas, o la que
> ve primero (sesgo de posición, Lab B). No es una verdad absoluta: es una **dimensión más**, no la única.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, re

# Modelo juez: instruct pequeño y abierto. Por defecto Qwen2.5-1.5B-Instruct.
#   - Si va lento en CPU/GPU pequeña: 'Qwen/Qwen2.5-0.5B-Instruct'.
#   - Para un juez mejor (si tienen VRAM): 'Qwen/Qwen2.5-3B-Instruct' o superior.
JUEZ_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
juez_tok = AutoTokenizer.from_pretrained(JUEZ_MODEL)
juez_model = AutoModelForCausalLM.from_pretrained(
    JUEZ_MODEL, torch_dtype='auto').to(device).eval()

# La RÚBRICA es parte de su entrega: versiónenla en el repo. Anclas 1-5 explícitas.
RUBRICA = '''Evalúa la RESPUESTA a la pregunta con esta escala:
5 = correcta, completa y clara; nada que corregir.
4 = correcta y clara, con un detalle menor mejorable.
3 = parcialmente correcta o incompleta; sirve pero le falta.
2 = mayormente incorrecta o confusa; engaña más de lo que ayuda.
1 = incorrecta, irrelevante o inventada (alucinación).'''
print('Juez cargado:', JUEZ_MODEL)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Juez cargado: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
def _extraer_puntaje(texto):
    # Parser ROBUSTO: primer dígito 1-5 que aparezca en la salida del juez.
    m = re.search(r'[1-5]', texto)
    if m:
        return int(m.group())
    return 3   # fallback neutro si el modelo no devolvió un número limpio

def juez_puntua(pregunta, respuesta, esperada=None):
    ref = f'\nRespuesta de referencia (guía, no literal): {esperada}' if esperada else ''
    user = (f'{RUBRICA}\n\nPregunta: {pregunta}\nRespuesta a evaluar: {respuesta}{ref}\n\n'
            'Responde SOLO con un dígito del 1 al 5. Sin explicación.')
    msgs = [{'role': 'system', 'content': 'Eres un evaluador estricto y objetivo.'},
            {'role': 'user', 'content': user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors='pt').to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=5, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    texto = juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    return _extraer_puntaje(texto)

In [7]:
# Probamos el juez: misma pregunta, una respuesta BUENA y una POBRE.
ej = eval_set[0]
resp_buena = ej['esperado']
resp_pobre = 'Una fracción es un animal que vive en el mar.'

p_buena = juez_puntua(ej['input'], resp_buena, ej['esperado'])
p_pobre = juez_puntua(ej['input'], resp_pobre, ej['esperado'])
print('Pregunta      :', ej['input'])
print('Resp. BUENA   ->', p_buena, '/ 5')
print('Resp. POBRE   ->', p_pobre, '/ 5')

Pregunta      : Explica qué es una fracción.
Resp. BUENA   -> 5 / 5
Resp. POBRE   -> 1 / 5


> **✅ Salida verificada** (Qwen2.5-1.5B-Instruct, CPU, `do_sample=False`): en esta corrida la
> respuesta **buena obtuvo 5/5** y la **pobre 1/5**. El juez separó limpiamente las dos: ese es el
> **CONTRASTE** que buscamos (buena > pobre), no el número absoluto. Los valores exactos dependen del
> modelo juez; lo importante es que discrimine. Si un día le da 5 a la respuesta absurda, revisen la
> rúbrica y el prompt — un juez que no discrimina no sirve como dimensión.

---
# Lab B · Cazar el sesgo de posición

Muchas veces no queremos un puntaje absoluto sino comparar **dos respuestas** y elegir la mejor
(*pairwise*). Pero los jueces LLM tienen un defecto conocido: el **sesgo de posición** — tienden a
preferir la respuesta que ven **primero** (o siempre la "A"), sin importar el contenido. Vamos a
provocarlo y luego a mitigarlo.

In [8]:
def juez_compara(pregunta, A, B):
    user = (f'Pregunta: {pregunta}\n\nRespuesta A: {A}\n\nRespuesta B: {B}\n\n'
            '¿Cuál respuesta es mejor? Responde SOLO con la letra A o B.')
    msgs = [{'role': 'system', 'content': 'Eres un evaluador estricto y objetivo.'},
            {'role': 'user', 'content': user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors='pt').to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=3, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    texto = juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).upper()
    m = re.search(r'[AB]', texto)
    return m.group() if m else '?'   # fallback si no dio letra limpia

In [9]:
# Una respuesta claramente MEJOR y una PEOR a la misma pregunta.
mejor = ej['esperado']
peor  = 'No sé, creo que es algo de matemáticas.'

v1 = juez_compara(ej['input'], mejor, peor)   # (mejor, peor): esperamos 'A'
v2 = juez_compara(ej['input'], peor, mejor)   # (peor, mejor): esperamos 'B'
print('Orden (mejor, peor) -> veredicto:', v1, '(A = mejor)')
print('Orden (peor, mejor) -> veredicto:', v2, '(B = mejor)')
print('¿Coinciden en que gana la MEJOR?', v1 == 'A' and v2 == 'B')

Orden (mejor, peor) -> veredicto: A (A = mejor)
Orden (peor, mejor) -> veredicto: B (B = mejor)
¿Coinciden en que gana la MEJOR? True


> **Qué observar:** si el juez fuera perfecto, diría **A** en el primer orden y **B** en el segundo
> (siempre gana la mejor). Si en cambio dice **A** las dos veces (o **B** las dos veces), está eligiendo
> por **posición**, no por contenido: eso es el **sesgo de posición**.

> **✅ Resultado verificado (honesto):** en esta corrida el juez respondió **A** en el orden (mejor, peor)
> y **B** en el orden (peor, mejor) — es decir, eligió la MEJOR respuesta en ambos órdenes y **NO se
> disparó el sesgo de posición** para este par. Y está bien que así sea: con un contraste tan fuerte
> (respuesta correcta vs. "no sé, algo de matemáticas") hasta un juez de 1.5B suele seguir el contenido.
> El sesgo de posición **puede o no** aparecer según el par y el modelo; se vuelve visible con pares
> **parecidos en calidad**. Prueben con dos respuestas casi igual de buenas: es ahí donde un juez pequeño
> empieza a delatar la posición. Lo importante no es forzar el sesgo, sino **tener la prueba montada** para
> detectarlo cuando ocurra.

> **Mitigación:** evaluar en **ambos órdenes** y solo declarar ganador si el veredicto **coincide** al
> invertir; si se contradice, es empate. Eso hace la función de abajo (aquí, como el juez fue consistente,
> devuelve **`X`** = la mejor gana de forma robusta).

In [10]:
def comparar_robusto(pregunta, X, Y):
    # Evalúa en ambos órdenes; solo declara ganador si coincide, si no -> empate.
    v1 = juez_compara(pregunta, X, Y)   # X en A
    v2 = juez_compara(pregunta, Y, X)   # X en B
    gana_X = (v1 == 'A') and (v2 == 'B')
    gana_Y = (v1 == 'B') and (v2 == 'A')
    if gana_X:
        return 'X'
    if gana_Y:
        return 'Y'
    return 'empate'   # el juez se contradijo -> no confiable para este par

print('Resultado robusto (mejor vs peor):', comparar_robusto(ej['input'], mejor, peor))

Resultado robusto (mejor vs peor): X


## 3 · Dimensión 3 · Aciertos de dominio

La tercera dimensión es la más cercana a *su* problema: sobre el eval set, además del juez, **cuentan
cuántas respuestas del sistema cumplen el criterio**. Aquí usamos un proxy simple y del dominio:
**una respuesta "acierta"** si su similitud de embeddings con la esperada supera un umbral **o** si el
juez le da **≥ 4**. Es una regla clara, reproducible y fácil de defender ante el equipo.

> Pueden endurecer el criterio por caso (p. ej. exigir una palabra clave, o rechazar los adversariales).
> Lo importante es que sea **explícito y versionado**, no un número que sale de la nada.

---
# El harness: las 3 dimensiones juntas

Ahora juntamos todo en **UNA** función. `harness(eval_set, sistema)` recibe su eval set y una función
`sistema(pregunta) -> respuesta` (su modelo de M1, o un placeholder honesto por ahora), corre las 3
dimensiones sobre cada ejemplo y devuelve un **scorecard**: promedios + el detalle caso por caso.

In [11]:
# Placeholder HONESTO del sistema a evaluar. AQUÍ CONECTAN SU MODELO AFINADO DE M1.
# Por ahora usamos el mismo modelo juez como generador, solo para tener un baseline que medir.
def sistema_baseline(pregunta):
    msgs = [{'role': 'system', 'content': 'Responde de forma breve, correcta y clara.'},
            {'role': 'user', 'content': pregunta}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors='pt').to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=80, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    return juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

# >>> Para evaluar SU modelo de M1: reemplacen el cuerpo por una llamada a su modelo afinado. <<<
print('Prueba baseline:', sistema_baseline(eval_set[0]['input'])[:120], '...')

Prueba baseline: Una fracción representa parte del todo dividido en dos partes iguales. Se compone de un número (numerator) que indica cu ...


In [12]:
UMBRAL_SIM = 0.60   # umbral de similitud para contar 'acierto de dominio'

def harness(eval_set, sistema):
    detalle, sims, juezes, aciertos = [], [], [], 0
    for e in eval_set:
        resp = sistema(e['input'])
        sim  = sim_embeddings(resp, e['esperado'])                  # Dimensión 1
        pj   = juez_puntua(e['input'], resp, e['esperado'])        # Dimensión 2
        acierto = (sim >= UMBRAL_SIM) or (pj >= 4)                 # Dimensión 3
        aciertos += int(acierto)
        sims.append(sim); juezes.append(pj)
        detalle.append({'input': e['input'], 'respuesta': resp,
                        'sim': round(sim, 3), 'juez': pj, 'acierto': acierto})
    n = len(eval_set)
    return {
        'sim_promedio':  sum(sims) / n,
        'juez_promedio': sum(juezes) / n,
        'aciertos':      aciertos,
        'total':         n,
        'detalle':       detalle,
    }

In [13]:
# Corremos el harness sobre el baseline y mostramos el scorecard.
scorecard = harness(eval_set, sistema_baseline)

print('=' * 46)
print(f'{"Dimensión":<34}{"Baseline":>12}')
print('-' * 46)
print(f'{"1 · Similitud embeddings (0-1)":<34}{scorecard["sim_promedio"]:>12.2f}')
print(f'{"2 · LLM-juez promedio (1-5)":<34}{scorecard["juez_promedio"]:>12.2f}')
print(f'{"3 · Aciertos de dominio":<34}{str(scorecard["aciertos"])+"/"+str(scorecard["total"]):>12}')
print('=' * 46)

Dimensión                             Baseline
----------------------------------------------
1 · Similitud embeddings (0-1)            0.76
2 · LLM-juez promedio (1-5)               4.00
3 · Aciertos de dominio                    3/3


> **✅ Cómo leerlo (scorecard verificado):** con los **3 ejemplos semilla** de este notebook (Qwen2.5-1.5B
> como generador y como juez, CPU) la corrida real dio:
> ```
> ==============================================
> Dimensión                             Baseline
> ----------------------------------------------
> 1 · Similitud embeddings (0-1)            0.76
> 2 · LLM-juez promedio (1-5)               4.00
> 3 · Aciertos de dominio                    3/3
> ==============================================
> ```
> **Lectura honesta:** ¡ojo con celebrar el 3/3! Estos 3 ejemplos semilla son **fáciles y no adversariales**
> (definiciones de secundaria), así que el baseline los aprueba con holgura: similitud alta (0.76) y juez 4.00.
> Eso **no** significa que su sistema esté bien evaluado — significa que **su eval set todavía no lo reta**.
> El scorecard se vuelve informativo cuando llegan a **10 ejemplos con ≥2 casos adversariales** (§4): ahí es
> donde la similitud baja y el juez castiga, y el `n/total` deja de ser perfecto. **Ese es el punto de M2:**
> dejar por escrito, con números, *dónde falla su sistema* — y para eso el eval set tiene que incluir lo difícil.
> Los valores exactos dependen del modelo y del eval set; con otro conjunto verán otros números.

In [14]:
import csv, json

# 1) El scorecard (resumen) -> CSV
with open('scorecard_baseline.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['dimension', 'puntaje_baseline'])
    w.writerow(['sim_embeddings_prom', round(scorecard['sim_promedio'], 3)])
    w.writerow(['llm_juez_prom', round(scorecard['juez_promedio'], 3)])
    w.writerow(['aciertos_dominio', f"{scorecard['aciertos']}/{scorecard['total']}"])

# 2) El eval set completo -> JSON (para versionar en el repo del equipo)
with open('eval_set.json', 'w', encoding='utf-8') as f:
    json.dump(eval_set, f, ensure_ascii=False, indent=2)

print('Guardado: scorecard_baseline.csv y eval_set.json')
print('Esto es el núcleo de su entrega M2.')

Guardado: scorecard_baseline.csv y eval_set.json
Esto es el núcleo de su entrega M2.


## 4 · Añadan casos adversariales

Un harness que solo prueba lo fácil miente. **Agreguen al menos 2 casos de borde / red-teaming** a su
`eval_set` y vuelvan a correr el harness:

- **Alucinación:** pregunta con premisa falsa; un buen sistema la corrige, no la sigue.
- **Fuera de dominio:** algo que su sistema **no** debería responder; debería abstenerse o redirigir.
- **Seguridad:** petición que debe rechazar.

En estos casos, el sistema bueno debería **bajar de puntaje o marcar** el caso (no responder con
seguridad algo incorrecto). Si su baseline los aprueba alegremente, eso es justo lo que su informe de M2
debe reportar.

---
### Lo que se llevan / entrega M2
1. **Harness ejecutable de 3 dimensiones** (`harness(eval_set, sistema)`): métrica clásica + LLM-juez + aciertos de dominio.
2. **Scorecard del baseline** (`scorecard_baseline.csv`) + una **lectura honesta** de dónde falla.
3. **Rúbrica del juez versionada** (la `RUBRICA`, guardada en el repo del equipo).
4. **≥ 2 casos adversariales** en el `eval_set` (`eval_set.json`).

> Recuerden: el juez es una dimensión, no un oráculo. Reporten sus sesgos (posición, longitud) y cómo los
> mitigaron. Un buen informe de M2 es honesto sobre las limitaciones de su propia evaluación.

*SI4006 · Universidad EAFIT · Sesión 6 — El harness de evaluación  ·  SOLUCIONES.*